<!-- RAG with PDF data extraction  -->

In [1]:
!pip install pypdf

In [2]:
import os

from dotenv import load_dotenv
load = load_dotenv(".env")

In [3]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    base_url="http://localhost:11434",
    model="qwen3:8b",
    temperature=0.5,
    num_predict=2202
)


In [4]:
# Ectracting PDF files
from langchain_community.document_loaders import PyPDFLoader

pdf1 = "attention.pdf"
pdf2 = "LLMForgetting.pdf"
pdf3 = "TestingAndEvaluatingLLM.pdf"
pdf4 = "user_Profile.pdf.pdf"

pdfFiles = [pdf1, pdf2, pdf3, pdf4]

documents = []

for pdf in pdfFiles:
    loader = PyPDFLoader(pdf)
    documents.extend(loader.load())

print(f"Total number of pages in all PDFs: {len(documents)}")

C:\Users\Girish Kulkarni\AppData\Local\Temp\ipykernel_860\3846132490.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Type': '/Font', '/Subtype': '/Type1', '/BaseFont': '/TAUCXQ+TimesNewRomanPSMT', '/Encoding': IndirectObject(419, 0, 2948146821936), '/FirstChar': 3, '/FontDescriptor': IndirectObject(420, 0, 2948146821936), '/LastChar': 31, '/Widths': [500, 333, 500, 278, 944, 333, 556, 500, 667, 500, 722, 389, 444, 722, 278, 611, 500, 722, 722, 500, 278, 444, 889, 500, 500, 500, 500, 250, 500]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Ty

Total number of pages in all PDFs: 262


In [5]:
# Text Splitting

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, add_start_index=True)

all_split_docs = text_splitter.split_documents(documents)

len(all_split_docs)  # Total number of chunks after splitting the documents


663

In [6]:
# Embedding the chunks

from langchain_core.embeddings import Embeddings
from langchain_ollama import OllamaEmbeddings

# Ollama can fail when a large document list is sent in one request.
ollama_embeddings = OllamaEmbeddings(model="nomic-embed-text")


class BatchedOllamaEmbeddings(Embeddings):
    def __init__(self, embedding_model, batch_size=8):
        self.embedding_model = embedding_model
        self.batch_size = batch_size

    def embed_documents(self, texts):
        vectors = []
        for start in range(0, len(texts), self.batch_size):
            batch = texts[start:start + self.batch_size]
            vectors.extend(self.embedding_model.embed_documents(batch))
        return vectors

    def embed_query(self, text):
        return self.embedding_model.embed_query(text)


embeddings = BatchedOllamaEmbeddings(ollama_embeddings)

vector_one = embeddings.embed_query(all_split_docs[0].page_content)
vector_two = embeddings.embed_query(all_split_docs[1].page_content)

print(len(vector_one))
print(len(vector_two))

768
768


In [7]:
# Vector store

from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=all_split_docs,
    embedding=embeddings,
    persist_directory="./chroma_langchain_db_v3",
)

In [8]:
# Retrieve relevant chunks

from langchain_chroma import Chroma

vector_store = Chroma(
    persist_directory="./chroma_langchain_db_v3",
    embedding_function=embeddings,
)

question = "What is my overall AI career direction?"
retrieved_docs = vector_store.similarity_search(question, k=3)

retrieved_docs

[Document(id='af76040c-a25e-4507-8506-5896f4de9521', metadata={'producer': 'ReportLab PDF Library - (opensource)', 'start_index': 825, 'title': 'User Knowledge Base for RAG', 'keywords': '', 'page': 5, 'trapped': '/False', 'subject': '(unspecified)', 'moddate': '2026-09-07T11:56:22+00:00', 'source': 'user_Profile.pdf.pdf', 'total_pages': 9, 'creationdate': '2026-09-07T11:56:22+00:00', 'page_label': '6', 'author': 'Generated from conversation context', 'creator': '(unspecified)'}, page_content="The user's broader AI learning roadmap includes agentic systems and tool integration.\n\x7f\nAI agents.\n\x7f\nTool/function calling.\n\x7f\nLangGraph workflows.\n\x7f\nMCP servers and clients.\n\x7f\nPlaywright MCP agents.\n\x7f\nJira MCP integration concepts.\n\x7f\nUsing AI systems to fetch QA tickets and comments.\n\x7f\nPotential automated daily reporting workflows.\nThe user is interested in combining AI systems with practical software-testing workflows rather than\nstudying LLMs only as th

In [9]:
# Generate an answer from the retrieved context

context = "\n\n".join(doc.page_content for doc in retrieved_docs)

prompt = f"""Answer the question using only the context below.
If the answer is not present in the context, say: I don't know based on the documents.

Context:
{context}

Question: {question}
Answer:"""

response = llm.invoke(prompt)
print(response.content)

Your overall AI career direction focuses on integrating AI systems with practical software-testing workflows, emphasizing testing and evaluation of AI systems rather than solely building them. You aim to specialize in areas like retrieval quality, hallucination detection, grounding evaluation, and tool/function-calling validation. Your skill development targets include:  
- **Software test automation** (Playwright, WDIO, Appium, TypeScript).  
- **CI/CD and performance testing** (k6).  
- **LLM application development** and **RAG systems**.  
- **Vector databases** (e.g., ChromaDB) and **local models** (e.g., Ollama).  
- **AI testing frameworks** for latency, cost, security, and regression testing.  

This combines your expertise in traditional testing with AI/ML technologies to create robust, evaluable AI-driven workflows.


In [10]:
# Create a profile-specific retriever

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 3,
        "filter": {"source": "user_Profile.pdf.pdf"},
    },
)

retriever.invoke("What programming languages are listed in my profile?")

[Document(id='124b245b-2de1-4f46-a7b9-15737d1cb3b3', metadata={'title': 'User Knowledge Base for RAG', 'producer': 'ReportLab PDF Library - (opensource)', 'page_label': '1', 'start_index': 815, 'trapped': '/False', 'page': 0, 'moddate': '2026-09-07T11:56:22+00:00', 'total_pages': 9, 'author': 'Generated from conversation context', 'creator': '(unspecified)', 'subject': '(unspecified)', 'creationdate': '2026-09-07T11:56:22+00:00', 'source': 'user_Profile.pdf.pdf', 'keywords': ''}, page_content='Programming languages: Java, JavaScript, TypeScript.\nCore automation background: Appium with Java and Page Object Model (POM). The user is also working with\nWebdriverIO, Appium v2, TypeScript, Playwright, CI/CD, reporting, and test architecture.\n2. Current Technical Direction\nThe user is moving toward a combined profile centered on advanced test automation plus AI\ntesting/evaluation and AI application development.\n\x7f\nAI testing/evaluation: approximately 70% of the intended AI focus.\n\x7

In [11]:
# Full RetrievalQA implementation

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

question = "What programming languages are listed in my profile?"

prompt = ChatPromptTemplate.from_template("""Use only the context below to answer the question.
If the answer is not in the context, say: I don't know based on the documents.
Keep the answer concise and do not add unsupported details.

Context:
{context}

Question: {question}
Answer:""")


def format_documents(documents):
    return "\n\n".join(document.page_content for document in documents)


retrieval_qa_chain = (
    {
        "context": retriever | format_documents,
        "question": lambda value: value,
    }
    | prompt
    | llm
    | StrOutputParser()
)

answer = retrieval_qa_chain.invoke(question)
print(answer)

The programming languages listed in your profile are Java, JavaScript, and TypeScript.


In [12]:
# RetrivalQA

